# SQL Results in pandas

*Load SQL query results into a DataFrame and compare equivalent grouped summaries.*


## Prepare the Sales Table

The sales table holds five prepared records in an in-memory
SQLite database. `sale_id` is a primary key, and the other fields store
the product, category, quantity, whole-dollar unit price, and month number.
`execute()` runs one SQL statement; `executemany()` binds each prepared
tuple to the `?` placeholders. `commit()` completes the insert transaction.


In [1]:
import sqlite3

import pandas as pd

connection = sqlite3.connect(":memory:")

connection.execute("""
CREATE TABLE sales (
    sale_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price INTEGER NOT NULL,
    month INTEGER NOT NULL
)
""")

sales_rows = [
    ("Mouse", "Accessory", 3, 25, 1),
    ("Keyboard", "Accessory", 2, 45, 1),
    ("Monitor", "Display", 1, 180, 2),
    ("Laptop Stand", "Accessory", 4, 30, 2),
    ("Tablet", "Device", 2, 220, 3),
]

connection.executemany("""
INSERT INTO sales (product_name, category, quantity, unit_price, month)
VALUES (?, ?, ?, ?, ?)
""", sales_rows)
connection.commit()

print("Prepared sales:")
for row in connection.execute("SELECT * FROM sales ORDER BY sale_id"):
    print(row)


Prepared sales:
(1, 'Mouse', 'Accessory', 3, 25, 1)
(2, 'Keyboard', 'Accessory', 2, 45, 1)
(3, 'Monitor', 'Display', 1, 180, 2)
(4, 'Laptop Stand', 'Accessory', 4, 30, 2)
(5, 'Tablet', 'Device', 2, 220, 3)


There are five sales, numbered 1 through 5. Mouse, Keyboard, and
Laptop Stand belong to Accessory; Monitor belongs to Display,
and Tablet belongs to Device.


## Read a Query Result into a DataFrame

`pd.read_sql_query(sql, connection)` executes the query and gives the
result columns their SQL names. The DataFrame has its own default index.
SQL's `ORDER BY` defines the order in which these records arrive.


In [2]:
sales_df = pd.read_sql_query("""
SELECT sale_id, product_name, category, quantity, unit_price, month
FROM sales
ORDER BY sale_id
""", connection)

print("Sales DataFrame:")
print(sales_df)
print("Shape:", sales_df.shape)


Sales DataFrame:
   sale_id  product_name   category  quantity  unit_price  month
0        1         Mouse  Accessory         3          25      1
1        2      Keyboard  Accessory         2          45      1
2        3       Monitor    Display         1         180      2
3        4  Laptop Stand  Accessory         4          30      2
4        5        Tablet     Device         2         220      3
Shape: (5, 6)


The query produces a DataFrame with five rows and six columns. `sale_id` remains a data column; the DataFrame index runs from 0 to 4.


## Calculate and Group in pandas

Add revenue from the loaded numeric columns, then group by category.
Editing this DataFrame changes the Python object; it does not issue an
SQL update to the database.


In [3]:
sales_df["revenue"] = sales_df["quantity"] * sales_df["unit_price"]
pandas_summary = sales_df.groupby("category", as_index=False, sort=True)["revenue"].sum()

print("Revenue per sale:")
print(sales_df[["product_name", "revenue"]])
print("pandas category summary:")
print(pandas_summary)


Revenue per sale:
   product_name  revenue
0         Mouse       75
1      Keyboard       90
2       Monitor      180
3  Laptop Stand      120
4        Tablet      440
pandas category summary:
    category  revenue
0  Accessory      285
1     Device      440
2    Display      180


Accessory totals 285, Device totals 440, and Display totals 180 dollars. Revenue is a derived column in the DataFrame.


## Let SQL Perform the Same Aggregation

SQL can calculate the grouped revenue before transferring the result to
pandas. Give the aggregate the same name and sort by category so the
two result tables can be compared directly with `.equals()`.


In [4]:
sql_summary = pd.read_sql_query("""
SELECT category, SUM(quantity * unit_price) AS revenue
FROM sales
GROUP BY category
ORDER BY category
""", connection)

print("SQL category summary:")
print(sql_summary)
print("Summaries match:", sql_summary.equals(pandas_summary))


SQL category summary:
    category  revenue
0  Accessory      285
1     Device      440
2    Display      180
Summaries match: True


The equality check is `True`. The SQL query transfers three summary rows, while the earlier detail query transfers all five records.


## Pass Query Parameters through pandas

`params` supplies values for the query placeholders. Filtering in SQL
limits which records become part of the returned DataFrame.


In [5]:
selected_df = pd.read_sql_query("""
SELECT product_name, quantity * unit_price AS revenue
FROM sales
WHERE category = ?
ORDER BY sale_id
""", connection, params=("Accessory",))

print("Accessory revenue:")
print(selected_df)


Accessory revenue:
   product_name  revenue
0         Mouse       75
1      Keyboard       90
2  Laptop Stand      120


Only Mouse, Keyboard, and Laptop Stand are returned, with revenue 75, 90, and 120 dollars.


## Close the Connection

`.close()` releases the database connection. This database lives only
in memory, so its tables disappear when the connection closes.


In [6]:
connection.close()
print("Connection closed.")


Connection closed.


Load query results with `read_sql_query`, then use familiar DataFrame methods. SQL can filter and aggregate before data reaches pandas.

[Reading SQL queries with pandas](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html)
